# MDA Data Preprocessing Pipeline

**Role:** Data Cleaning & Processing (Wei Jie)  
**Input:** Raw MDA `.txt` files extracted from SEC 10-K/10-Q filings (`../MDA/`)  

**Primary Outputs** (schema aligned to the full project workflow):

| File | Granularity | Downstream Consumer |
|------|------------|--------------------|
| `processed/corpus_metadata.csv` | 1 row per filing | Both tasks — provides the base DataFrame |
| `processed/sentences.csv` | 1 row per sentence | **Sentiment Analysis** & **Topic Modelling** (JOIN key: `sentence_id`, `doc_id`) |
| `processed/topic_tokens.csv` | 1 row per filing | **Topic Modelling** only — lemmatized, phrase-detected token lists |

---

## Full Project Workflow Context

```
corpus_metadata.csv                       sentences.csv
(doc_id, company, ticker,          (sentence_id, doc_id, company, ticker,
 year, quarter, mda_text)           quarter, sentence, is_redundant, is_fls)
         │                                        │
         ▼                                        ├─────────────────────┐
  Topic Modelling                                 ▼                     ▼
  (LDA / BERTopic)                      Sentiment Analysis       Topic Modelling
  document-topic dist.                  (FinBERT / SVM)          (sentence topics)
         │                                        │
         └──────────────── JOIN on sentence_id ───┘
                                        │
                           (company × quarter × topic) matrix
                                        │
                                    Dashboard
```

## Pipeline Steps

```
Step 1 │ Load & Parse Filenames      → corpus DataFrame with metadata
Step 2 │ Ticker Mapping              → add ticker symbol per company
Step 3 │ Text Cleaning               → remove headers, encoding artefacts, whitespace
Step 4 │ Financial Table Removal     → strip numerical tables from prose
Step 5 │ Sentence Segmentation       → split each document into sentences
Step 6 │ Redundant Sentence Filter   → rule-based removal of boilerplate / noise
Step 7 │ Forward-Looking Statement   → flag speculative sentences
Step 8 │ NLP Preprocessing           → tokenize, lemmatize, POS tag, stopword removal
Step 9 │ Phrase Detection            → bigram/trigram collocations (topic modelling)
Step 10│ Export & Validate           → save all outputs, run sanity checks
```

## Setup & Imports

In [7]:
# Install required packages if not already present
# !pip install spacy gensim tqdm
!python -m spacy download en_core_web_sm
# !pip3 install matplotlib

# !pip install --upgrade numpy matplotlib
# !pip install scipy==1.11.4



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 31.6 MB/s eta 0:00:0000:010:01

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [1]:
import os
import re
import json
import warnings
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
from gensim.models.phrases import Phrases, ENGLISH_CONNECTOR_WORDS
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
tqdm.pandas()

MDA_DIR    = Path("../MDA")
DATA_DIR   = Path("../Data Preparation")
OUTPUT_DIR = Path("processed")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"MDA files found: {len(list(MDA_DIR.glob('*.txt')))}")

MDA files found: 18183


---
## Step 1: Load & Parse Filenames

**What:** Read every `.txt` file from `MDA/` and parse structured metadata from the filename.

**Rationale:** Filenames encode three fields — `{Company}_{FormType}_{YYYY-MM-DD}_MDA.txt`. We extract these into a structured DataFrame so that every subsequent row (sentence, topic, sentiment score) can be traced back to a specific company, filing type, and date — which is essential for the time-series analysis and dashboard aggregations downstream.

We also compute the **quarter label** (e.g. `Q1 2020`) at this stage because the dashboard groups everything by `(company, quarter)`. The full date is kept for the out-of-time train/test split.

In [2]:
def parse_filename(fname: str) -> dict:
    """
    Parse MDA filename into structured fields.
    Format: {Company}_{FormType}_{YYYY-MM-DD}_MDA.txt
    e.g.    NVIDIA_10-Q_2020-05-27_MDA.txt
    """
    stem = fname.replace("_MDA.txt", "")
    date_match = re.search(r"(\d{4}-\d{2}-\d{2})$", stem)
    if not date_match:
        return {"company": None, "form_type": None, "filing_date": None}

    date_str  = date_match.group(1)
    remainder = stem[: date_match.start()].rstrip("_")

    form_match = re.search(r"_(10-[KQ])$", remainder)
    if not form_match:
        return {"company": remainder.replace("_", " "), "form_type": None, "filing_date": date_str}

    form_type = form_match.group(1)
    company   = remainder[: form_match.start()].replace("_", " ")

    return {"company": company, "form_type": form_type, "filing_date": pd.to_datetime(date_str)}


records = []
for fpath in tqdm(sorted(MDA_DIR.glob("*.txt")), desc="Loading files"):
    meta = parse_filename(fpath.name)
    meta["filename"]  = fpath.name
    meta["raw_text"]  = fpath.read_text(encoding="utf-8", errors="replace")
    records.append(meta)

corpus = pd.DataFrame(records)

# Temporal fields the dashboard groups by
corpus["year"]          = corpus["filing_date"].dt.year
corpus["quarter_num"]   = corpus["filing_date"].dt.quarter
corpus["quarter_label"] = "Q" + corpus["quarter_num"].astype(str) + " " + corpus["year"].astype(str)

# Assign a stable doc_id (integer index) — this is the JOIN key used across all downstream outputs
corpus = corpus.reset_index(drop=True)
corpus.insert(0, "doc_id", corpus.index)

print(f"Total documents:  {len(corpus):,}")
print(f"Unique companies: {corpus['company'].nunique()}")
print(f"Form types:       {corpus['form_type'].value_counts().to_dict()}")
print(f"Date range:       {corpus['filing_date'].min().date()} → {corpus['filing_date'].max().date()}")
corpus[["doc_id","company","form_type","filing_date","quarter_label"]].head()

Loading files:   0%|          | 0/18183 [00:00<?, ?it/s]

Total documents:  18,183
Unique companies: 473
Form types:       {'10-Q': 13566, '10-K': 4460}
Date range:       2010-01-08 → 2025-12-23


,doc_id,company,form_type,filing_date,quarter_label
0,0,1-800-PetMeds,10-K,2020-05-26,Q2.0 2020.0
1,1,1-800-PetMeds,10-K,2021-05-25,Q2.0 2021.0
2,2,1-800-PetMeds,10-K,2022-05-24,Q2.0 2022.0
3,3,1-800-PetMeds,10-K,2023-05-23,Q2.0 2023.0
4,4,1-800-PetMeds,10-K,2024-06-14,Q2.0 2024.0


---
## Step 2: Ticker Mapping

**What:** Join a ticker symbol (e.g. `NVDA`, `AAPL`) onto every document.

**Rationale:** The dashboard schema requires a `ticker` column — it is used for portfolio lookups (*"What is the portfolio sentiment for {NVDA, MSFT, AAPL}?"*). Tickers are also a cleaner, unambiguous company identifier than the full name, which can vary across filings (e.g. `Alphabet (Google)` vs. `Alphabet`).

We load the ticker reference table from `TechCompanyByMarketCap_withCIK.csv` and fuzzy-match company names. Unmatched companies will have `ticker = NaN` — these can still be processed but will be excluded from dashboard portfolio views.

In [3]:
ticker_ref = pd.read_csv(DATA_DIR / "TechCompanyByMarketCap_withCIK.csv").drop_duplicates(subset="Name")
# Keep only the columns we need
ticker_ref = ticker_ref[["Name", "ticker"]].rename(columns={"Name": "ref_name", "ticker": "ticker"})

# Build a lookup dict: lowercase company name → ticker
# Handles common name variations (e.g. "Alphabet (Google)" → "GOOG")
ticker_lookup = {row.ref_name.lower().strip(): row.ticker for row in ticker_ref.itertuples()}

# Also add manual overrides for names that appear differently in filenames vs. the CSV
MANUAL_OVERRIDES = {
    "alphabet": "GOOG",
    "google": "GOOG",
    "meta platforms": "META",
    "meta": "META",
    "facebook": "META",
}
ticker_lookup.update(MANUAL_OVERRIDES)


def map_ticker(company_name: str) -> str | None:
    if pd.isna(company_name):
        return None
    key = company_name.lower().strip()
    # Exact match first
    if key in ticker_lookup:
        return ticker_lookup[key]
    # Partial match: check if any ref name is contained in the company name or vice versa
    for ref_key, ticker in ticker_lookup.items():
        if ref_key in key or key in ref_key:
            return ticker
    return None


corpus["ticker"] = corpus["company"].apply(map_ticker)

matched   = corpus["ticker"].notna().sum()
unmatched = corpus["ticker"].isna().sum()
print(f"Ticker matched:   {matched:,} documents ({matched/len(corpus)*100:.1f}%)")
print(f"Ticker unmatched: {unmatched:,} documents")

if unmatched > 0:
    print("\nTop unmatched company names (add to MANUAL_OVERRIDES if needed):")
    print(corpus[corpus["ticker"].isna()]["company"].value_counts().head(20).to_string())

Ticker matched:   17,242 documents (94.8%)
Ticker unmatched: 941 documents

Top unmatched company names (add to MANUAL_OVERRIDES if needed):
company
Booking Holdings  Booking com      65
Fair Isaac  FICO                   64
SS C Technologies                  64
Box  Inc                           63
Jack Henry  amp  Associates        63
QXO  Inc                           56
Strategy    MicroStrategy          44
Alpha  amp  Omega Semiconductor    44
UCT  Ultra Clean Holdings          43
Stem  Inc                          43
Hims  amp  Hers Health             26
Agora io                           24
SoundThinking    ShotSpotter       24
CarParts com                       24
Odysight ai                        24
Instacart  Maplebear Inc           22
Nerdy  Inc                         21
BigBear ai                         20
Sohu com                           14
MNTN  Inc                          13


---
## Step 3: Text Cleaning

**What:** Apply targeted cleaning rules to remove artefacts that survived the SEC filing extraction.

**Rationale:**

| Artefact | Example | Why remove |
|----------|---------|------------|
| Residual MD&A header | `ITEM 2. MANAGEMENTS DISCUSSION AND ANALYSIS...` | Repeated boilerplate, not content |
| Encoding replacement chars | `\ufffd` | Byte-level decode failures — not real text |
| Repeated punctuation runs | `......`, `- - -` | Table border formatting from HTML-to-text conversion |
| Excess whitespace / blank lines | multiple spaces, 4+ newlines | Reduces input size; prevents tokenizers treating whitespace as tokens |

In [4]:
_RE_HEADER       = re.compile(
    r"^\s*ITEM\s+\d+[A-Z]?\.?\s+MANAGEMENT[S']?S?\s+DISCUSSION[\s\S]{0,120}?OPERATIONS?\.?",
    re.IGNORECASE | re.MULTILINE,
)
_RE_REPLACE_CHAR = re.compile(r"\ufffd+")
_RE_REPEAT_PUNCT = re.compile(r"([.\-|*=#_]){3,}")
_RE_WHITESPACE   = re.compile(r"[ \t\xa0]+")
_RE_BLANK_LINES  = re.compile(r"\n{3,}")


def clean_text(text: str) -> str:
    text = _RE_HEADER.sub("", text)        # 1. Remove MD&A section header
    text = _RE_REPLACE_CHAR.sub(" ", text) # 2. Remove encoding artefacts
    text = _RE_REPEAT_PUNCT.sub(" ", text) # 3. Remove table border punctuation
    text = _RE_WHITESPACE.sub(" ", text)   # 4. Normalise intra-line whitespace
    text = _RE_BLANK_LINES.sub("\n\n", text) # 5. Collapse excessive blank lines
    return text.strip()


corpus["cleaned_text"] = corpus["raw_text"].progress_apply(clean_text)

raw_len   = corpus["raw_text"].str.len().mean()
clean_len = corpus["cleaned_text"].str.len().mean()
print(f"Avg raw length:     {raw_len:,.0f} chars")
print(f"Avg cleaned length: {clean_len:,.0f} chars  ({(raw_len-clean_len)/raw_len*100:.1f}% reduced)")

# Spot-check one document
sample = corpus[corpus["company"] == "NVIDIA"].iloc[0]
print("\n=== RAW (first 400 chars) ===")
print(sample["raw_text"][:400])
print("\n=== CLEANED (first 400 chars) ===")
print(sample["cleaned_text"][:400])

  0%|          | 0/18183 [00:00<?, ?it/s]

Avg raw length:     66,662 chars
Avg cleaned length: 66,587 chars  (0.1% reduced)

=== RAW (first 400 chars) ===
Managements Discussion and Analysis of Financial Condition and Results of Operations. The consolidated statements of operations data for the years ended January 31, 2010, January 25, 2009 and January 27, 2008 and the consolidated balance sheet data as of January 31, 2010 and January 25, 2009 have been derived from and should be read in conjunction with our audited consolidated financial statements

=== CLEANED (first 400 chars) ===
Managements Discussion and Analysis of Financial Condition and Results of Operations. The consolidated statements of operations data for the years ended January 31, 2010, January 25, 2009 and January 27, 2008 and the consolidated balance sheet data as of January 31, 2010 and January 25, 2009 have been derived from and should be read in conjunction with our audited consolidated financial statements


---
## Step 4: Financial Table Removal

**What:** Detect and remove passages that are dense numerical tables.

**Rationale:** MD&A sections contain both narrative prose *and* inline financial tables. Tables are noise for both downstream tasks:
- **Sentiment Analysis**: `Revenue $ 4,130,162 $ 4,280,159 $ 3,997,930` has no sentiment — it will add false signal.
- **Topic Modelling**: Isolated numbers and column headers will pollute topic word distributions.

**Detection heuristic:** A line is classified as a *table row* if it has ≥3 financial numbers (`$` amounts or large comma-separated integers) within a short span (< 300 chars). We remove contiguous *blocks* of 2+ such rows — this avoids stripping single inline statistics that are part of prose sentences.

In [5]:
_RE_FIN_NUMBER = re.compile(
    r"\$\s?[\d,]+(?:\.\d+)?|\(?[\d]{1,3}(?:,[\d]{3})+(?:\.\d+)?\)?"
)


def is_table_row(line: str) -> bool:
    return len(line) < 300 and len(_RE_FIN_NUMBER.findall(line)) >= 3


def remove_tables(text: str) -> str:
    lines = text.split("\n")
    flags = [is_table_row(l) for l in lines]
    keep  = []
    i = 0
    while i < len(lines):
        if flags[i]:
            j = i
            while j < len(lines) and flags[j]:
                j += 1
            if (j - i) < 2:        # single table-like line → likely inline stat, keep
                keep.append(lines[i])
            # else: block of 2+ rows → discard
            i = j
        else:
            keep.append(lines[i])
            i += 1
    return "\n".join(keep)


corpus["mda_text"] = corpus["cleaned_text"].progress_apply(remove_tables)

after_len = corpus["mda_text"].str.len().mean()
print(f"Avg after table removal: {after_len:,.0f} chars  ({(clean_len-after_len)/clean_len*100:.1f}% removed)")

  0%|          | 0/18183 [00:00<?, ?it/s]

Avg after table removal: 66,587 chars  (0.0% removed)


---
## Step 5: Sentence Segmentation

**What:** Split each document into individual sentences.

**Rationale:** According to the workflow architecture, both Topic Modelling and Sentiment Analysis operate on the same sentence-level data — they run in parallel on the same sentence corpus and are later joined on `sentence_id`. This makes sentence segmentation the most critical single step in the pipeline: it defines the unit of analysis for the entire project.

We use **spaCy's statistical sentence boundary detector** rather than a simple period-split because MD&A text contains:
- Abbreviations ending in periods (`U.S.`, `Inc.`, `No.`, `Corp.`)
- Decimal numbers (`$4.1 billion`, `16.2%`)
- Numbered list items (`1. Revenue grew...`)

A naive period-split would fragment all of these incorrectly, producing broken sentence fragments that confuse both FinBERT and topic models.

In [8]:
nlp_sent = spacy.load("en_core_web_sm", disable=["ner"])
nlp_sent.max_length = 2_000_000

MIN_SENT_LEN = 30    # chars — discard header fragments, page numbers, lone captions
MAX_SENT_LEN = 1500  # chars — very long "sentences" are malformed table rows that slipped through


def segment_sentences(row: pd.Series) -> list[dict]:
    doc = nlp_sent(row["mda_text"])
    sents = []
    for sent in doc.sents:
        text = sent.text.strip()
        if MIN_SENT_LEN <= len(text) <= MAX_SENT_LEN:
            sents.append({
                "doc_id":         row["doc_id"],
                "company":        row["company"],
                "ticker":         row["ticker"],
                "form_type":      row["form_type"],
                "filing_date":    row["filing_date"],
                "year":           row["year"],
                "quarter_num":    row["quarter_num"],
                "quarter_label":  row["quarter_label"],
                "sentence":       text,
            })
    return sents


print("Segmenting sentences (may take a few minutes for 18k documents)...")
all_sentences = []
for _, row in tqdm(corpus.iterrows(), total=len(corpus), desc="Sentence splitting"):
    all_sentences.extend(segment_sentences(row))

sentences_df = pd.DataFrame(all_sentences).reset_index(drop=True)
sentences_df.insert(0, "sentence_id", sentences_df.index)  # globally unique sentence key

print(f"\nTotal sentences: {len(sentences_df):,}")
print(f"Avg sentences per document: {len(sentences_df)/len(corpus):.0f}")
sentences_df.head(3)

Segmenting sentences (may take a few minutes for 18k documents)...


Sentence splitting:   0%|          | 0/18183 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Sentence length distribution
sent_lens = sentences_df["sentence"].str.len()
print(f"Median sentence length: {sent_lens.median():.0f} chars")
print(f"Mean sentence length:   {sent_lens.mean():.0f} chars")

plt.figure(figsize=(10, 3))
plt.hist(sent_lens.clip(upper=600), bins=60, color="steelblue", edgecolor="white")
plt.title("Sentence Length Distribution (clipped at 600 chars)")
plt.xlabel("Length (chars)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sentence_length_dist.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Step 6: Redundant Sentence Filtering (Rule-Based)

**What:** Mark sentences as redundant using deterministic rules. Redundant sentences are **kept in the DataFrame** with `is_redundant=True` — downstream teams can choose their own inclusion/exclusion threshold — but they are excluded from the default analysis.

**Rationale:** After sentence splitting, several categories of non-informative sentences remain that would degrade model quality:

| Category | Example | Why redundant |
|----------|---------|---------------|
| Cross-references | *"See Note 3 to our financial statements"* | Navigation pointer, no content |
| Exhibit references | *"Incorporated herein by reference to Exhibit 99.1"* | Legal filing artefact |
| Safe Harbor disclaimer | *"Forward-looking statements are not guarantees of future performance"* | Identical boilerplate in every filing |
| Sentence too short | *"Total."*, *"See above."* | Fragment, no signal |
| Table caption remnants | *"(In thousands, except per share data)"* | Missed table row |
| Pure numerical sentences | *"2019 2018 2017"* | Column headers that slipped through |

These rules are intentionally conservative — we only flag sentences we are confident are noise.

In [ ]:
# --- Rule definitions ---

# Cross-reference and exhibit patterns
_RE_XREF = re.compile(
    r"(see note|see item|see part|see exhibit|incorporated herein|refer to note|refer to item"
    r"|incorporated by reference|please refer|as described (?:above|below)|\bExhibit \d+)",
    re.IGNORECASE,
)

# Safe harbor / forward-looking disclaimer boilerplate (the disclaimer paragraph itself)
_RE_SAFE_HARBOR = re.compile(
    r"(forward.looking statements? (?:are|is|can be|provide|include)"
    r"|not guarantee[sd]? of future"
    r"|actual results? (?:may|might|could) differ"
    r"|assumes? no obligation to (?:revise|update)"
    r"|identified by words such as)",
    re.IGNORECASE,
)

# Table caption patterns (unit annotations that survived table removal)
_RE_TABLE_CAPTION = re.compile(
    r"^\s*\(?in (thousands|millions|billions)?[,;]? (?:except|unless)",
    re.IGNORECASE,
)

# Sentences that are overwhelmingly numeric (≥ 60% tokens are numbers/punctuation)
def is_mostly_numeric(text: str) -> bool:
    tokens = text.split()
    if not tokens:
        return True
    numeric = sum(1 for t in tokens if re.fullmatch(r"[\d,.$%()\-]+", t))
    return numeric / len(tokens) >= 0.6


def flag_redundant(sentence: str) -> bool:
    """
    Returns True if the sentence is considered non-informative noise.
    Conservative: only flag sentences we are highly confident add no signal.
    """
    if len(sentence.strip()) < 40:          # Very short fragments
        return True
    if _RE_XREF.search(sentence):           # Cross-references to other sections
        return True
    if _RE_SAFE_HARBOR.search(sentence):    # Safe harbor boilerplate
        return True
    if _RE_TABLE_CAPTION.search(sentence):  # Table unit captions
        return True
    if is_mostly_numeric(sentence):         # Pure number rows
        return True
    return False


sentences_df["is_redundant"] = sentences_df["sentence"].progress_apply(flag_redundant)

n_redundant = sentences_df["is_redundant"].sum()
print(f"Redundant sentences flagged: {n_redundant:,} ({n_redundant/len(sentences_df)*100:.1f}%)")
print(f"Clean sentences remaining:   {(~sentences_df['is_redundant']).sum():,}")

# Show a sample of what was flagged
print("\nSample redundant sentences:")
for s in sentences_df[sentences_df["is_redundant"]]["sentence"].sample(5, random_state=42).values:
    print(f"  • {s[:120]}")

---
## Step 7: Forward-Looking Statement (FLS) Flagging

**What:** Tag sentences that contain speculative / forward-looking language with `is_forward_looking=True`.

**Rationale:** This addresses the *Speculative Language* text challenge from the project proposal. Forward-looking sentences express *intent or expectation* about the future, not factual sentiment about past performance. Mixing them with factual sentiment sentences can bias classifiers — *"We expect revenue to decline"* is negative, but it is management's projection, not a reported outcome.

The `is_forward_looking` flag lets downstream teams:
- **Exclude** FLS when training sentiment models on factual performance
- **Analyse separately** to distinguish *guidance tone* (what management says will happen) vs. *reported tone* (what actually happened)

In [ ]:
# Loughran-McDonald forward-looking word list + common MD&A phrasing patterns
_FLS_PATTERN = re.compile(
    r"\b(will|expect[s]?|expected|anticipate[s]?|anticipated|intend[s]?|intended"
    r"|plan[s]?|planned|project[s]?|projected|forecast[s]?|forecasted"
    r"|believe[s]?|believed|estimate[s]?|estimated|may|might|could|should|would"
    r"|future|potential|outlook|guidance|going forward|in the future"
    r"|we expect|we anticipate|we believe|we intend|we plan)\b",
    re.IGNORECASE,
)

sentences_df["is_forward_looking"] = sentences_df["sentence"].str.contains(
    _FLS_PATTERN, regex=True
)

fls_rate = sentences_df["is_forward_looking"].mean()
print(f"Forward-looking sentences: {sentences_df['is_forward_looking'].sum():,} ({fls_rate*100:.1f}%)")

# Cross-tabulation: how many clean (non-redundant) sentences are FLS?
clean = sentences_df[~sentences_df["is_redundant"]]
print(f"\nAmong clean sentences ({len(clean):,} total):")
print(f"  Factual (not FLS):       {(~clean['is_forward_looking']).sum():,}")
print(f"  Forward-looking (FLS):   {clean['is_forward_looking'].sum():,}")

---
## Step 8: NLP Preprocessing

**What:** Tokenize, POS-tag, lemmatize, and remove stopwords from each clean sentence.

**Rationale — two separate token representations are produced:**

| Column | Contains | Used by |
|--------|---------|--------|
| `sentence` | Original text (unchanged) | **FinBERT / SVM** — transformer and TF-IDF models need the raw sentence |
| `pos_tags` | `[(word, POS), ...]` | **Aspect Extraction** with GLiNER — POS context improves NER accuracy |
| `tokens_lemma` | Lemmatized, stopword-removed alpha tokens | **LDA / NMF** — bag-of-words topic models need clean vocabulary |

**Why lemmatize?** MD&A text is inflected: `declining`, `declined`, `decline` are all the same concept. Without lemmatization they occupy separate vocabulary slots, fragmenting the signal that LDA would otherwise concentrate on one topic word.

**Why remove domain stopwords?** Words like `company`, `quarter`, `management`, `operations` appear in virtually every MD&A sentence. They dominate TF-IDF weights and pollute LDA topics without adding topical signal. We add a financial-domain stoplist on top of spaCy's standard English one.

In [ ]:
nlp_full = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp_full.max_length = 2_000_000

FINANCE_STOPWORDS = {
    "company", "companies", "business", "period", "year", "quarter",
    "fiscal", "financial", "result", "results", "operation", "operations",
    "management", "discussion", "analysis", "following", "pursuant",
    "form", "filing", "report", "annual", "quarterly", "sec", "edgar",
    "item", "table", "note", "including", "certain", "various",
    "also", "however", "therefore", "whereas", "herein", "thereof",
    "therein", "hereof", "hereto", "thereto",
}
for word in FINANCE_STOPWORDS:
    nlp_full.vocab[word].is_stop = True


# Only run NLP preprocessing on clean (non-redundant) sentences for efficiency
# Redundant sentences get empty lists
clean_mask = ~sentences_df["is_redundant"]
clean_texts = sentences_df.loc[clean_mask, "sentence"].tolist()

tokens_lemma_out = [None] * len(sentences_df)
pos_tags_out     = [None] * len(sentences_df)
clean_indices    = sentences_df.index[clean_mask].tolist()

print(f"Running NLP on {len(clean_texts):,} clean sentences (batched)...")

for i, doc in enumerate(tqdm(
    nlp_full.pipe(clean_texts, batch_size=512, disable=["ner", "parser"]),
    total=len(clean_texts), desc="NLP preprocessing",
)):
    tokens_lemma, pos_tags = [], []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        pos_tags.append((token.lower_, token.pos_))
        if token.is_alpha and not token.is_stop and len(token.lemma_) > 2:
            tokens_lemma.append(token.lemma_.lower())

    idx = clean_indices[i]
    tokens_lemma_out[idx] = tokens_lemma
    pos_tags_out[idx]     = pos_tags

# Fill redundant sentences with empty lists
tokens_lemma_out = [t if t is not None else [] for t in tokens_lemma_out]
pos_tags_out     = [t if t is not None else [] for t in pos_tags_out]

sentences_df["tokens_lemma"] = tokens_lemma_out
sentences_df["pos_tags"]     = pos_tags_out

print("Done.")
print("Sample tokens_lemma:", sentences_df[clean_mask].iloc[0]["tokens_lemma"][:12])
print("Sample pos_tags:    ", sentences_df[clean_mask].iloc[0]["pos_tags"][:8])

---
## Step 9: Phrase Detection (Bigrams & Trigrams)

**What:** Detect frequently co-occurring word pairs/triples and fuse them into single tokens.

**Rationale:** Financial text is dense with multi-word concepts that should be treated as single semantic units by topic models:
- `supply_chain`, `gross_margin`, `operating_income`, `research_and_development`
- `free_cash_flow`, `earnings_per_share`, `capital_expenditure`

Without phrase detection, LDA/NMF would spread these across separate uninformative single-word topics. Joining them produces human-readable topics like *{supply_chain, disruption, inventory, constraint}* rather than *{supply, chain, disruption, inventory}*.

We train gensim's `Phrases` model on the lemmatized token lists from all clean, non-FLS sentences (factual prose only). Two passes:
1. **Bigrams**: `supply` + `chain` → `supply_chain`
2. **Trigrams**: `free` + `cash` + `flow` → `free_cash_flow`

The phrase models are saved so the topic modelling team can apply them consistently to new documents.

In [ ]:
# Train only on factual, non-redundant sentences for cleaner phrase discovery
train_mask   = clean_mask & ~sentences_df["is_forward_looking"]
train_tokens = sentences_df.loc[train_mask, "tokens_lemma"].tolist()

print(f"Training phrase models on {len(train_tokens):,} factual sentences...")

bigram_model  = Phrases(train_tokens,  min_count=20, threshold=10, connector_words=ENGLISH_CONNECTOR_WORDS)
bigram_sents  = [bigram_model[sent] for sent in train_tokens]
trigram_model = Phrases(bigram_sents,  min_count=10, threshold=10, connector_words=ENGLISH_CONNECTOR_WORDS)

# Apply phrase models to ALL clean sentences (including FLS — topic modelling team may want them)
all_tokens = sentences_df["tokens_lemma"].tolist()
sentences_df["tokens_phrases"] = [
    trigram_model[bigram_model[sent]] if sent else []
    for sent in tqdm(all_tokens, desc="Applying phrases")
]

# Show top discovered phrases
all_phrases = [
    t for sent in sentences_df["tokens_phrases"]
    for t in sent if "_" in t
]
print("\nTop 25 discovered financial phrases:")
for phrase, count in Counter(all_phrases).most_common(25):
    print(f"  {phrase:<40} {count:>8,}")

In [ ]:
# Save the phrase models so the topic modelling team can apply them to new documents
bigram_model.save(str(OUTPUT_DIR / "bigram_model"))
trigram_model.save(str(OUTPUT_DIR / "trigram_model"))
print("Phrase models saved to:", OUTPUT_DIR.resolve())

---
## Step 10: Export & Validate

**What:** Save three output files and run schema validation checks.

**Output schema summary:**

### `corpus_metadata.csv` — document-level
| Column | Type | Description |
|--------|------|-------------|
| `doc_id` | int | Unique document identifier (JOIN key) |
| `company` | str | Company name |
| `ticker` | str | Stock ticker (e.g. `NVDA`) |
| `form_type` | str | `10-K` or `10-Q` |
| `filing_date` | date | Full filing date |
| `year` | int | Filing year |
| `quarter_num` | int | 1–4 |
| `quarter_label` | str | e.g. `Q1 2020` |
| `mda_text` | str | Cleaned, de-tabled MDA text |

### `sentences.csv` — sentence-level (primary output)
| Column | Type | Description |
|--------|------|-------------|
| `sentence_id` | int | Globally unique sentence key (JOIN key) |
| `doc_id` | int | Parent document |
| `company` | str | Company name |
| `ticker` | str | Stock ticker |
| `form_type` | str | `10-K` or `10-Q` |
| `filing_date` | date | Filing date |
| `year` | int | Filing year |
| `quarter_num` | int | 1–4 |
| `quarter_label` | str | e.g. `Q1 2020` |
| `sentence` | str | Original sentence text |
| `is_redundant` | bool | Rule-based noise flag |
| `is_forward_looking` | bool | FLS flag |
| `tokens_lemma` | JSON list | Lemmatized tokens, stopwords removed |
| `tokens_phrases` | JSON list | Phrase-fused token list |
| `pos_tags` | JSON list | `[(word, POS), ...]` |

### `topic_tokens.csv` — document-level, for topic modelling
| Column | Type | Description |
|--------|------|-------------|
| `doc_id` | int | Document identifier |
| `company`, `ticker`, `quarter_label`, ... | — | Metadata fields |
| `doc_tokens` | JSON list | All phrase-fused tokens for this document (factual sentences only) |
| `token_count` | int | Number of tokens |

In [ ]:
# ── Output 1: Corpus metadata ──────────────────────────────────────────────
corpus_out = corpus[[
    "doc_id", "company", "ticker", "form_type",
    "filing_date", "year", "quarter_num", "quarter_label",
    "mda_text"
]]
corpus_out.to_csv(OUTPUT_DIR / "corpus_metadata.csv", index=False)
print(f"Saved corpus_metadata.csv  — {len(corpus_out):,} rows")


# ── Output 2: Sentence-level dataset ──────────────────────────────────────
sentences_out = sentences_df[[
    "sentence_id", "doc_id", "company", "ticker", "form_type",
    "filing_date", "year", "quarter_num", "quarter_label",
    "sentence", "is_redundant", "is_forward_looking",
    "tokens_lemma", "tokens_phrases", "pos_tags",
]].copy()

for col in ["tokens_lemma", "tokens_phrases", "pos_tags"]:
    sentences_out[col] = sentences_out[col].apply(json.dumps)

sentences_out.to_csv(OUTPUT_DIR / "sentences.csv", index=False)
print(f"Saved sentences.csv        — {len(sentences_out):,} rows")


# ── Output 3: Topic modelling document corpus ──────────────────────────────
# Aggregate phrase tokens per document (factual sentences only)
RARE_THRESHOLD = 5
all_phrase_counts = Counter(
    t for sent in sentences_df.loc[~sentences_df["is_redundant"] & ~sentences_df["is_forward_looking"], "tokens_phrases"]
    for t in sent
)
rare_tokens = {t for t, c in all_phrase_counts.items() if c < RARE_THRESHOLD}

factual_mask = ~sentences_df["is_redundant"] & ~sentences_df["is_forward_looking"]

doc_tokens_map = (
    sentences_df[factual_mask]
    .groupby("doc_id")["tokens_phrases"]
    .apply(lambda seqs: [t for sent in seqs for t in sent if t not in rare_tokens])
)

topic_df = corpus_out.drop(columns=["mda_text"]).merge(
    doc_tokens_map.reset_index().rename(columns={"tokens_phrases": "doc_tokens"}),
    on="doc_id", how="left"
)
topic_df["doc_tokens"]  = topic_df["doc_tokens"].apply(lambda x: x if isinstance(x, list) else [])
topic_df["token_count"] = topic_df["doc_tokens"].apply(len)

MIN_DOC_TOKENS = 50
before = len(topic_df)
topic_df = topic_df[topic_df["token_count"] >= MIN_DOC_TOKENS].reset_index(drop=True)
print(f"Dropped {before - len(topic_df)} docs with < {MIN_DOC_TOKENS} tokens after cleaning")

topic_df["doc_tokens"] = topic_df["doc_tokens"].apply(json.dumps)
topic_df.to_csv(OUTPUT_DIR / "topic_tokens.csv", index=False)
print(f"Saved topic_tokens.csv     — {len(topic_df):,} rows")

### Validation

In [ ]:
print("=" * 60)
print("VALIDATION REPORT")
print("=" * 60)

val_corpus = pd.read_csv(OUTPUT_DIR / "corpus_metadata.csv", parse_dates=["filing_date"])
val_sents  = pd.read_csv(OUTPUT_DIR / "sentences.csv",       parse_dates=["filing_date"])
val_topics = pd.read_csv(OUTPUT_DIR / "topic_tokens.csv",    parse_dates=["filing_date"])

print("\n[corpus_metadata.csv]")
print(f"  Rows:            {len(val_corpus):,}")
print(f"  Null values:     {val_corpus.isnull().sum().sum()}")
print(f"  Ticker coverage: {val_corpus['ticker'].notna().mean()*100:.1f}%")
print(f"  Date range:      {val_corpus['filing_date'].min().date()} → {val_corpus['filing_date'].max().date()}")
print(f"  Quarter sample:  {val_corpus['quarter_label'].value_counts().head(3).to_dict()}")

print("\n[sentences.csv]")
print(f"  Rows:              {len(val_sents):,}")
print(f"  Null sentences:    {val_sents['sentence'].isnull().sum()}")
print(f"  Redundant:         {val_sents['is_redundant'].mean()*100:.1f}%")
print(f"  Forward-looking:   {val_sents['is_forward_looking'].mean()*100:.1f}%")
print(f"  Usable (clean+factual): {(~val_sents['is_redundant'] & ~val_sents['is_forward_looking']).sum():,}")
# Verify JSON columns parse
assert isinstance(json.loads(val_sents.loc[val_sents['tokens_lemma'] != '[]'].iloc[0]['tokens_lemma']), list)
print(f"  JSON columns parse: OK")

print("\n[topic_tokens.csv]")
tc = val_topics["token_count"]
print(f"  Rows:           {len(val_topics):,}")
print(f"  Min tokens/doc: {tc.min()}")
print(f"  Avg tokens/doc: {tc.mean():.0f}")
print(f"  Max tokens/doc: {tc.max()}")
assert isinstance(json.loads(val_topics.iloc[0]['doc_tokens']), list)
print(f"  JSON columns parse: OK")

print("\n" + "=" * 60)
print("All outputs valid. Outputs saved to:", OUTPUT_DIR.resolve())
print("=" * 60)

---
## Downstream Usage Guide

### Sentiment Analysis team

```python
import pandas as pd, json

df = pd.read_csv("DataProcessing/processed/sentences.csv", parse_dates=["filing_date"])

# Use only clean, factual sentences
factual = df[~df["is_redundant"] & ~df["is_forward_looking"]]

# FinBERT / SVM: use raw `sentence` column
X = factual["sentence"].tolist()

# Time-based train/test split (no data leakage)
train = factual[factual["year"] < 2022]
test  = factual[factual["year"] >= 2022]

# GLiNER aspect extraction: POS tags available
factual["pos_tags"] = factual["pos_tags"].apply(json.loads)

# After labelling, JOIN with topic output on sentence_id + doc_id:
# final = factual.merge(topic_sentence_df, on=["sentence_id", "doc_id"])
# final.groupby(["company", "quarter_label", "topic"]).agg({"sentiment_score": "mean"})
```

### Topic Modelling team

```python
import pandas as pd, json
from gensim.corpora import Dictionary
from gensim.models.phrases import Phraser

# Document-level tokens for LDA / NMF
df = pd.read_csv("DataProcessing/processed/topic_tokens.csv", parse_dates=["filing_date"])
df["doc_tokens"] = df["doc_tokens"].apply(json.loads)
token_lists = df["doc_tokens"].tolist()

# Build gensim Dictionary + Corpus
dct    = Dictionary(token_lists)
dct.filter_extremes(no_below=5, no_above=0.85)
corpus = [dct.doc2bow(doc) for doc in token_lists]

# BERTopic: use raw mda_text from corpus_metadata.csv
meta     = pd.read_csv("DataProcessing/processed/corpus_metadata.csv")
raw_docs = meta["mda_text"].tolist()

# Apply saved phrase models to new documents:
# bigram  = Phraser.load("DataProcessing/processed/bigram_model")
# trigram = Phraser.load("DataProcessing/processed/trigram_model")
```